# Timeseries Singleton Fill¶

Open netcdf file, evaluate time-step and make it consistent

1min, 10min, 1hr, etc

In [1]:
import xarray as xr
import datetime

In [26]:
####### edit these values to point to the file in question
user_data_dir = '/Users/bell/ecoraid/2022/Moorings/22bs5a/final_data_cf/'
filename = '22bs5a_s37_0053m.nc'

# 2*interp_gap
sample_t = {'1T':'2T',
            '30T':'1H',
            '1H':'2H',
           }

interp_gap = sample_t['1H'] #designed sample frequency


### load data and determine timebase

In [27]:
xdf = xr.load_dataset(user_data_dir+filename)
xdf

<xarray.Dataset> Size: 275kB
Dimensions:       (time: 8591, depth: 1, latitude: 1, longitude: 1)
Coordinates:
  * time          (time) datetime64[ns] 69kB 2022-09-07 ... 2023-08-30T22:00:00
  * depth         (depth) float64 8B 53.0
  * latitude      (latitude) float64 8B 59.91
  * longitude     (longitude) float64 8B -171.7
Data variables:
    temperature   (time, depth, latitude, longitude) float64 69kB 0.2798 ... ...
    conductivity  (time, depth, latitude, longitude) float64 69kB 2.693 ... 2...
    salinity      (time, depth, latitude, longitude) float64 69kB 31.89 ... 3...
Attributes: (12/33)
    Latitude_DegMMddW:                      59 54.747 N
    Longitude_DegMMddN:                     171 43.160 W
    MooringID:                              22BS-5A
    platform_deployment_date:               2022-09-07T00:00:00Z
    platform_deployment_cruise_name:        AQ2201
    platform_recovery_date:                 2023-08-30T22:00:00Z
    ...                                     ...
    InstrumentSerialNumber:                 2331
    InstrumentType:                         SBE-37
    date_created:                           2023-10-19T11:35:59Z
    date_modified:                          2023-10-19 11:38:53.980269
    QC_indicator:                           ProbablyGood
    history:                                Trimmed to deployment.\nQC'd: 202...

In [28]:
(xdf.time.diff(dim='time') * 1e-9 ).mean()

<xarray.DataArray 'time' ()> Size: 8B
array(3600, dtype='timedelta64[ns]')

In [29]:
xdf.resample({'time':'1h'}).mean()

<xarray.Dataset> Size: 275kB
Dimensions:       (time: 8591, depth: 1, latitude: 1, longitude: 1)
Coordinates:
  * time          (time) datetime64[ns] 69kB 2022-09-07 ... 2023-08-30T22:00:00
  * depth         (depth) float64 8B 53.0
  * latitude      (latitude) float64 8B 59.91
  * longitude     (longitude) float64 8B -171.7
Data variables:
    temperature   (time, depth, latitude, longitude) float64 69kB 0.2798 ... ...
    conductivity  (time, depth, latitude, longitude) float64 69kB 2.693 ... 2...
    salinity      (time, depth, latitude, longitude) float64 69kB 31.89 ... 3...
Attributes: (12/33)
    Latitude_DegMMddW:                      59 54.747 N
    Longitude_DegMMddN:                     171 43.160 W
    MooringID:                              22BS-5A
    platform_deployment_date:               2022-09-07T00:00:00Z
    platform_deployment_cruise_name:        AQ2201
    platform_recovery_date:                 2023-08-30T22:00:00Z
    ...                                     ...
    InstrumentSerialNumber:                 2331
    InstrumentType:                         SBE-37
    date_created:                           2023-10-19T11:35:59Z
    date_modified:                          2023-10-19 11:38:53.980269
    QC_indicator:                           ProbablyGood
    history:                                Trimmed to deployment.\nQC'd: 202...

In [28]:
xdf.to_netcdf(user_data_dir+filename,format="NETCDF3_CLASSIC",encoding={'time':{'units':'days since 1900-01-01'}})

/var/folders/6r/n_mpcj7d7pdf9ncqvjy5vb8c0000jm/T/ipykernel_75704/3629195037.py:1: UserWarning: Times can't be serialized faithfully to int64 with requested units 'days since 1900-01-01'. Resolution of 'hours' needed. Serializing times to floating point instead. Set encoding['dtype'] to integer dtype to serialize to int64. Set encoding['dtype'] to floating point dtype to silence this warning.
  xdf.to_netcdf(user_data_dir+filename,format="NETCDF3_CLASSIC",encoding={'time':{'units':'days since 1900-01-01'}})
